In [ ]:
from google.cloud import bigquery  # Conector oficial do banco de dados analítico (BigQuery) da Google.
from google.oauth2 import service_account  # Mecanismo de autenticação por chave/arquivo JSON de conta de serviço.

# Manipulação e Análise de Dados
import pandas as pd 
import pandas_gbq  

# 3. CONFIGURAÇÃO DE FONTES DE DADOS E INFRAESTRUTURA DE NUVEM
# ==============================================================================

# --- FONTE 1: REPOSITÓRIO REMOTO (GITHUB) ---
# URL base onde estão armazenados arquivos CSV públicos contendo dados históricos de filmes.
url_github_movies_dataset = "https://media.githubusercontent.com/media/lucas-aulas/dataset-movies/refs/heads/main"

# --- FONTE 2: AMBIENTE DE LEITURA (PROJETO COMPARTILHADO DO PROFESSOR) ---
# ID do projeto na Google Cloud de onde leremos uma tabela institucional de bilheterias.
fonte_gcp_project_id = "meu-projeto-eng-dados"

# --- DESTINO: AMBIENTE DE ESCRITA (PROJETO PRIVADO DO ALUNO) ---
# Caminho local do arquivo JSON que contém as credenciais de segurança (chave privada).
# ATENÇÃO: Nunca envie esta chave para o GitHub público! Ela equivale à senha do seu banco de dados.
meu_service_account_file = r"C:\Users\1597779\data_project-template\secrets\meu-projeto-engdados-c67a1176f58c.json"
meu_gcp_project_id = "meu-projeto-engdados"  # ID do seu projeto pessoal criado no GCP.

# Inicialização da autenticação e do cliente BigQuery
# Transforma o arquivo JSON em um objeto de credenciais aceito pelo Google.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# O objeto 'client' funciona como a nossa ponte ativa de conexão/comunicação com a nuvem do Google.
bigquery_client = bigquery.Client(credentials=gcp_credentials)

In [ ]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# ------------------------------------------------------------------------------
# Fonte: Camada Bronze - BigQuery
# Objetivo:
#   Carregar dados financeiros e comerciais de filmes da Marvel e DC.
#
# Dataset:
#   bronze.boxoffice_dc_marvel
# ------------------------------------------------------------------------------

df_boxoffice_dc_marvel = bigquery_client.query("""
    SELECT *
    FROM bronze.boxoffice_dc_marvel
    """).to_dataframe(create_bqstorage_client=False)

In [ ]:
df_boxoffice_dc_marvel.head(5)  

In [ ]:
df_imdb_filmes = bigquery_client.query("""
    SELECT *
    FROM bronze.imdb_filmes
    """).to_dataframe(create_bqstorage_client=False)


In [ ]:
df_imdb_filmes.head()

In [ ]:
df_complete_movie_info = (
    pd.merge(
        df_boxoffice_dc_marvel,
        df_imdb_filmes,
        left_on=["FILM", "YEAR"],
        right_on=["TITLE", "YEAR"],
        how="outer",
    )
    .sort_values(by="YEAR", ascending=False)
    .drop_duplicates(keep="last")
)

In [ ]:
# ==============================================================================
# 6. FILTRAGEM DE FRANQUIAS (MARVEL E DC)
# ==============================================================================

# Mantendo apenas filmes pertencentes às franquias:
#   - Marvel
#   - DC
#
# Isso elimina registros de outros universos cinematográficos presentes no dataset.

df_complete_dc_marvel = df_complete_movie_info[
    (df_complete_movie_info["FRANCHISE"] == "Marvel") | (df_complete_movie_info["FRANCHISE"] == "DC")
]

In [ ]:
# ==============================================================================
# 7. REMOÇÃO DE COLUNAS REDUNDANTES E DESNECESSÁRIAS
# ==============================================================================

# Eliminando colunas:
#   - duplicadas após o merge,
#   - auxiliares,
#   - pouco relevantes para análises finais,
#   - ou já representadas por outras métricas equivalentes.

df_complete_dc_marvel = df_complete_dc_marvel.drop(
    columns=[
        "25X_PROD",
        "BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA",
        "BOX_OFFICE_GROSS_WORLDWIDE",
        "BUDGET_y",
        "DURATION",
        "LENGTH",
        "MPAA_RATING",
        "TITLE",
        "YEAR",
    ]
).reset_index(drop=True)

In [ ]:
df_complete_dc_marvel.sort_values(by="RATING_IMDB", ascending=False).head(2)

In [ ]:
df_tratado = df_complete_dc_marvel.copy()

In [ ]:
money_cols = [
    "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
    "BUDGET_x",
    "INFLATION_ADJUSTED_BUDGET",
    "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
    "GROSS_OPENING_WEEKEND",
    "GROSS_US_CANADA",
    "GROSS_WORLD_WIDE",
]

for col in money_cols:
    df_tratado[col] = (
        df_tratado[col]
        .astype(str)
        .str.replace("$", "")
        .str.replace(",", "")
        .replace("None", 0)
        .astype(float)
    )

In [ ]:
percent_cols = ["DOMESTIC"]

for col in percent_cols:
    df_tratado[col] = df_tratado[col].astype(str).str.replace("%", "").astype(float)

In [ ]:
date_cols = ["US_RELEASE_DATE"]

for col in date_cols:
    df_tratado[col] = pd.to_datetime(df_tratado[col], format="%m/%d/%Y")

In [ ]:
int_cols = [
    "VOTE",
    "WIN",
    "OSCAR",
    "NOMINATION",
    "ROTTEN_TOMATOES_CRITIC_SCORE",
]

for col in int_cols:
    df_tratado[col] = df_tratado[col].fillna(0).astype(int)

In [ ]:
df_tratado["YEAR"] = df_tratado["US_RELEASE_DATE"].dt.year

In [ ]:
df_tratado["GENRE"].str.split(",").str[0].str.strip()

df_tratado["COUNTRY_ORIGIN"] = df_tratado["COUNTRY_ORIGIN"].str.split(",").str[0].str.strip()

df_tratado["LANGUAGE"] = df_tratado["LANGUAGE"].str.split(",").str[0].str.strip()

df_tratado["PRODUCTION_COMPANY"] = df_tratado["PRODUCTION_COMPANY"].str.split(",").str[0].str.strip()

df_tratado["WRITER"] = df_tratado["WRITER"].str.split(",").str[0].str.strip()

In [ ]:
df_tratado = df_tratado[sorted(df_tratado.columns)].drop_duplicates()

In [ ]:
pandas_gbq.to_gbq(
   df_tratado,
   project_id=meu_gcp_project_id,
   destination_table="silver.movies_dc_marvel",
   if_exists="replace",
   credentials=gcp_credentials, 
)